# Introduction

Using the solutions from previous exercises, copy-paste the definitions of the created functions. Next, retrieve the organizations data and save them to the `yougov.json` file.

### Hints
- scraping the website takes some time, we recommend using the `tqdm` library to track the progress,
- remember that until now, when we worked in separate notebooks, it was necessary to accept cookies every time - now it is possible that we will need to modify this.
- reading and writing to JSON: `Day 4 - API -> JSON`

In [ ]:
# import required libraries here
from selenium import webdriver
from time import sleep
from tqdm import tqdm

# Selenium sam pronalazi, preuzima i pokreće odgovarajući ChromeDriver
browser = webdriver.Chrome()

In [ ]:
def get_organization_list():
    sleep(1)
    orgs_container = browser.find_element('css selector', '.rankings-entities-list')
    orgs = orgs_container.find_elements("tag name", 'li')

    results = []
    for org in orgs:
        name = org.find_elements('tag name', 'span')[2].text
        link = org.find_element('tag name', 'a').get_attribute('href')
        
        record = {
            'name': name,
            'link': link
        }

        results.append(record)    
    return results  

In [ ]:
# Drugi nacin za pronalazak detailed data

def get_organization_detailed_data(url):
    browser.get(url)

    # Kratko čekanje da se sadržaj učita (jednostavan način sinkronizacije)
    sleep(1)

    # Pronađi kontejner koji sadrži header-e
    header_lines = browser.find_element('css selector', '.entity-header-lines')

    # Unutar kontejnera pronađi sva polja s nazivima/etiketama (npr. "Fame", "Popularity")
    lines_labels = header_lines.find_elements('css selector', '.label')

    # Unutar kontejnera pronađi sva polja s vrednostima (npr. "97%", "81%")
    lines_values = header_lines.find_elements('css selector', '.value')

    # prazan rečnik za parove
    headers = {}

    # Prođi kroz sve nađene etikete; koristimo indeks da uparimo s odgovarajućom vrednošću
    for i in range(len(lines_labels)):
        # Dohvati tekst etikete na poziciji i
        label = lines_labels[i].text

        # Dohvati tekst odgovarajuće vrednosti na istoj poziciji i
        value = lines_values[i].text

        # Spremi u rečnik: ključ = etiketa, vrednost = tekst vrednosti
        headers[label] = value

    return headers

In [ ]:
# get the list of organizations here
browser.implicitly_wait(10)

url = 'https://yougov.co.uk/ratings/politics/popularity/charities-organisations/all'
browser.get(url)

sleep(1)
coockies_btn = browser.find_element('id', 'onetrust-accept-btn-handler')
coockies_btn.click()

organization_list = get_organization_list()
organization_list

In [ ]:
# get all the required data here
detailed_data = []  # prazna lista u koju ćemo spremati sve zapise (record)
for org in tqdm(organization_list):  # prolazi kroz svaku organizaciju uz prikaz napretka
    # Eksplicitno dohvatimo polja iz rječnika umjesto oslanjanja na redoslijed values()
    organization = org['name']  # ime organizacije
    url = org['link']  # link na stranicu organizacije
    data = get_organization_detailed_data(url)  # poziva funkciju koja dohvaća detaljne podatke za URL
    record = {  # sastavlja rječnik s podacima za trenutnu organizaciju
        'name': organization,
        'link': url,
        'data': data
    }

    detailed_data.append(record)  # dodaje sastavljeni zapis u listu `detailed_data`
detailed_data  # u Jupyteru/Notebooku: prikazuje sadržaj liste (sakupljeni podaci)

In [ ]:
browser.close()

In [ ]:
# save the data to yougov.json file here
import json

file_name = 'yougov.json'
with open(file_name, 'w') as f:
    json.dump(detailed_data, f)